In [66]:
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv, GATConv,GATv2Conv

class GraphEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, edge_dim):
        super(GraphEncoder, self).__init__()
        self.out_channels=out_channels
        self.conv1 = GATv2Conv(in_channels, hidden_channels, edge_dim=edge_dim, heads=4, concat=True)
        # Second GATv2Conv layer
        self.conv2 = GATv2Conv(hidden_channels * 4, out_channels, edge_dim=edge_dim, heads=1, concat=False)

    def forward(self, x, edge_index, edge_attr):
        # x: Node features
        # edge_index: Edge connections
        # edge_attr: Edge features
        x = self.conv1(x, edge_index,edge_attr)
        x = torch.relu(x)
        x = self.conv2(x, edge_index,edge_attr)
        return x

class TemporalModel(nn.Module):
    def __init__(self, graph_encoder, hidden_dim, num_layers,out_channel):
        super(TemporalModel, self).__init__()
        self.graph_encoder = graph_encoder
        self.rnn = nn.LSTM(input_size=graph_encoder.out_channels, 
                           hidden_size=hidden_dim, 
                           num_layers=num_layers, 
                           batch_first=True)
        self.fc = nn.Linear(hidden_dim,out_channel)  # Predict for the ego vehicle

    def forward(self, graph_sequence):
        # graph_sequence: List of graphs over time
        graph_embeddings = []
        for graph in graph_sequence:
            x, edge_index, edge_attr = graph.x, graph.edge_index, graph.edge_attr
            graph_embedding = self.graph_encoder(x, edge_index, edge_attr)
            graph_embeddings.append(graph_embedding)
        
        # Stack embeddings over time
        graph_embeddings = torch.stack(graph_embeddings, dim=1)  # Shape: [batch_size, seq_len, embedding_dim]
        
        # Pass through RNN
        rnn_out, _ = self.rnn(graph_embeddings)
        
        # Predict for the ego vehicle (assume it's the first node in the graph)
        ego_vehicle_embedding = rnn_out[:, -1, :]  # Last timestamp
        output = self.fc(ego_vehicle_embedding)
        return output

    

In [67]:
import torch
from torch_geometric.data import Data

# Define node features (4 nodes, 2 features each)
x = torch.tensor([
    [1.0, 2.0],  # Node 0 features
    [3.0, 4.0],  # Node 1 features
    [5.0, 6.0],  # Node 2 features
    [7.0, 8.0]   # Node 3 features
], dtype=torch.float)

# Define edge connections (source, target)
edge_index = torch.tensor([
    [0, 1, 2, 3, 1],  # Source nodes
    [1, 2, 3, 0, 3]   # Target nodes
], dtype=torch.long)

# Define edge features (5 edges, 1 feature each)
edge_attr = torch.tensor([
    [0.1],  # Edge 0->1
    [0.2],  # Edge 1->2
    [0.3],  # Edge 2->3
    [0.4],  # Edge 3->0
    [0.5]   # Edge 1->3
], dtype=torch.float)

# Create the graph object
graph = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

graph_sequence=[]
graph_sequence.append(graph)
graph_sequence.append(graph)

encoder=GraphEncoder(2,128,16,1)
model=TemporalModel(encoder,16,1,1)
model.forward(graph_sequence)

tensor([[-0.1741],
        [-0.1763],
        [-0.1732],
        [-0.1729]], grad_fn=<AddmmBackward0>)

In [ ]:
from torch_geometric.nn import MessagePassing
import torch.nn.functional as F

class EdgeFeatureGNN(MessagePassing):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(EdgeFeatureGNN, self).__init__(aggr='add')  # Aggregation method: 'add', 'mean', or 'max'
        self.node_mlp = nn.Sequential(
            nn.Linear(in_channels + hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, out_channels)
        )
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels)
        )

    def forward(self, x, edge_index, edge_attr):
        # x: Node features [num_nodes, in_channels]
        # edge_index: Edge connections [2, num_edges]
        # edge_attr: Edge features [num_edges, edge_feature_dim]
        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i, x_j, edge_attr):
        # x_i: Features of target nodes
        # x_j: Features of source nodes
        # edge_attr: Edge features
        edge_features = self.edge_mlp(edge_attr)  # Transform edge features
        return torch.cat([x_j, edge_features], dim=-1)  # Concatenate source node and edge features

    def update(self, aggr_out, x):
        # aggr_out: Aggregated messages
        # x: Original node features
        return self.node_mlp(torch.cat([x, aggr_out], dim=-1))  # Combine with original node features
    

class GraphEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, edge_feature_dim):
        super(GraphEncoder, self).__init__()
        self.gnn1 = EdgeFeatureGNN(in_channels, hidden_channels, hidden_channels)
        self.gnn2 = EdgeFeatureGNN(hidden_channels, hidden_channels, out_channels)

    def forward(self, x, edge_index, edge_attr):
        x = self.gnn1(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.gnn2(x, edge_index, edge_attr)
        return x